# IMP Demo — Collaborator Guide

Interactive sandbox over `src/`. Use this notebook to try **new data, features, and models** before promoting anything to `main.py`.

## Prerequisites

- Put required parquets under `data/` (see `config/default.py`). Optional demo macro: `data/macro_daily.parquet`.
- Run cells **top to bottom**. After changing CONFIG or §3 / §5+, re-run from that section through §7.
- If you edit `src/`, restart the kernel (or re-run the import cell) before fitting again.

## Where to edit

| Goal | Edit here | Notes |
| ---- | --------- | ----- |
| Target / split / which families | **§0a** `TARGET`, `FEATURE_COLUMNS`, `FAMILIES`, … | Comment/uncomment feature names to A/B. |
| Built-in lin / tr / nn knobs | **§0b / 0c / 0d** | Only families listed in `FAMILIES` actually run. |
| New daily table (e.g. macro) | **§3** — merge onto `df_align` by `Date` | Market-level: `Date` only. Stock-level: `Date` + `TickerId`. |
| Engineered features | **§3** (on `df_align`, before event table) | Then add column names to `FEATURE_COLUMNS`. |
| Event / preprocess / metrics | **§4–§7** | Prefer not to change wiring unless you know the pipeline. |
| Custom model (demo-only) | **§5+a / 5+b / 5+c** | Enable `"custom"` in `FAMILIES`. Leave **Wire-up** alone. |

## Feature checklist

1. Build or merge columns on `df_align` in **§3** (DecisionDate-safe; no lookahead).
2. List those names in **§0a** `FEATURE_COLUMNS` (missing names → `KeyError` in §4).
3. Re-run **§4 → §7** (not only fit).

## Custom model checklist

1. Uncomment `"custom"` in **§0a** `FAMILIES` (alone or with `lin` / `tr` / `nn`).
2. Edit only **§5+a / 5+b / 5+c** (hyperparams + fitter). Roles mirror lin / tr / nn.
3. Re-run **§5+ → §7**. Do not edit the Wire-up cell unless adding a new model id.

## Demo vs `main.py`

- §3 merges, engineered features, and `"custom"` stay in this notebook unless you later move them into `src/` / `config/` / `model/`.
- Batch baseline: `python main.py` with the same `FAMILIES` idea for `lin` / `tr` / `nn` only.


## 0. Config

Shared knobs (including which families to run), then hyperparameter cells for
`LIN` / `TR` / `NN`. Those family cells always define settings; only `FAMILIES`
in **0a** decides what actually runs.


In [237]:
from __future__ import annotations

import importlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import src.train as _train
importlib.reload(_train)

from src.report import print_notebook_config
from src.train import resolve_demo_setup


In [238]:
pd.set_option("display.width", None) 
pd.set_option("display.expand_frame_repr", False)  
pd.set_option("display.max_columns", None)

### 0a. Shared


In [239]:
# Shared across families (target / split / importance).
TARGET = "ret_ann_mkt"
# Original 8: QScoreTrendLong, QScoreSpeedLong, QScoreLong, analystRatingsStrongBuy, analystRatingsBuy, analystRatingsHold, analystRatingsSell, analystRatingsStrongSell
# Engineered in §3++ (from LastPrice / MarketCap).
FEATURE_COLUMNS = (
    ["QScoreTrendLong", "QScoreSpeedLong", "QScoreLong", "analystRatingsStrongBuy", "analystRatingsBuy", "analystRatingsHold", "analystRatingsSell", "analystRatingsStrongSell"]
    # + ["vix", "tnx", "dxy"]
    # + ["mom_21d", "vol_21d", "log_mcap"]
)

SPLIT_COLUMN = "DecisionDate"
TRAIN_END = "2022-12-31"
VAL_END = "2023-12-31"
IMPUTE_STRATEGY = "median"
RANDOM_STATE = 42
FEATURE_IMPORTANCE = False
FEATURE_IMPORTANCE_REPEATS = 10

# Which families to run (same idea as main.py FAMILIES).
# "custom" is demo-only; define CUSTOM + fit_custom in §5+.
FAMILIES = (
    "lin",
    # "tr",
    # "nn",
    # "custom",
)


### 0b. Linear

Hyperparameters for the linear family (used when `"lin"` is in `FAMILIES`).
Defaults match `config/config_lin.py`.


In [240]:
LIN = {
    "MODEL_TYPES": ("ols", "ridge", "lasso"),
    "ALPHA": 1.0,
    "ALPHA_GRID": (0.01, 0.1, 1.0, 10.0, 100.0),
    "STANDARDIZE": True,
}


### 0c. Trees

Hyperparameters for the tree family (used when `"tr"` is in `FAMILIES`).
Defaults match `config/config_tr.py`.


In [241]:
TR = {
    "MODEL_TYPES": (
        "decision_tree",
        "random_forest",
        "hist_gradient_boosting",
    ),
    "N_ESTIMATORS": 200,
    "MAX_DEPTH": 4,
    "MAX_DEPTH_GRID": (2, 3, 4, 6, 8),
    "LEARNING_RATE": 0.05,
    "MIN_SAMPLES_LEAF": 20,
    "EARLY_STOPPING_PATIENCE": 10,
    "STANDARDIZE": False,
}


### 0d. Neural

Hyperparameters for the neural family (used when `"nn"` is in `FAMILIES`).
Defaults match `config/config_nn.py`.


In [242]:
NN = {
    "MODEL_TYPES": ("mlp_shallow", "mlp", "mlp_deep"),
    "HIDDEN_SIZES": {
        "mlp_shallow": (64,),
        "mlp": (64, 32),
        "mlp_deep": (128, 64, 32),
    },
    "ALPHA": 1e-4,
    "ALPHA_GRID": (1e-5, 1e-4, 1e-3, 1e-2),
    "LR": 1e-3,
    "BATCH_SIZE": 64,
    "MAX_ITER": 200,
    "EARLY_STOPPING_PATIENCE": 8,
    "STANDARDIZE": True,
}


### Resolve Active Families And Data Paths


In [243]:
setup = resolve_demo_setup(FAMILIES, lin=LIN, tr=TR, nn=NN, root=ROOT)
ACTIVE_FAMILIES = list(setup.active_families)
STANDARDIZE = setup.standardize
DATA_DIR = setup.data_dir


In [244]:
print_notebook_config(
    target=TARGET,
    feature_columns=FEATURE_COLUMNS,
    split_column=SPLIT_COLUMN,
    train_end=TRAIN_END,
    val_end=VAL_END,
    impute_strategy=IMPUTE_STRATEGY,
    random_state=RANDOM_STATE,
    feature_importance=FEATURE_IMPORTANCE,
    feature_importance_repeats=FEATURE_IMPORTANCE_REPEATS,
    standardize=STANDARDIZE,
    active_families=ACTIVE_FAMILIES,
)


------------------------------------------------------------
CONFIG
------------------------------------------------------------
ACTIVE_FAMILIES  ['lin']
TARGET            ret_ann_mkt
FEATURE_COLUMNS   (8)
                  - QScoreTrendLong
                  - QScoreSpeedLong
                  - QScoreLong
                  - analystRatingsStrongBuy
                  - analystRatingsBuy
                  - analystRatingsHold
                  - analystRatingsSell
                  - analystRatingsStrongSell
SPLIT_COLUMN      DecisionDate
TRAIN_END         2022-12-31
VAL_END           2023-12-31
IMPUTE_STRATEGY   median
STANDARDIZE (step5 preview) True
RANDOM_STATE      42
FEATURE_IMPORTANCE False
FEATURE_IMPORTANCE_REPEATS 10

[lin]
  MODEL_TYPES            ('ols', 'ridge', 'lasso')
  ALPHA                  1.0
  ALPHA_GRID             (0.01, 0.1, 1.0, 10.0, 100.0)
  STANDARDIZE            True


## 1. Load And Merge

Load the three Parquet tables and build the daily panel `df_all` via `src.prepare` (`load_stock`, `load_earnings`, `load_sentiment`, `merge_panel`).

In [245]:
from src.prepare import load_earnings, load_sentiment, load_stock, merge_panel

In [246]:
stock_data = load_stock(DATA_DIR)
stock_data.head()

          | 0/242507

,TickerId,ShortName,InstrumentName,Date,LastPrice,High,Open,Low,Volume,CurrencyId,QScoreTrendLong,QScoreSpeedLong,QScoreLong,MarketCap
0,1256,ABNB:XNGS,AIRBNB INC-A,2024-12-31,131.4100,133.1900,132.4600,131.2700,2312564.0000,840,-0.8571,0.5714,-0.2857,56884312691.9000
1,1256,ABNB:XNGS,AIRBNB INC-A,2024-12-30,131.8100,132.6100,131.3100,129.8500,2458835.0000,840,-1.1429,0.4286,-0.7143,57057463327.9000
2,1256,ABNB:XNGS,AIRBNB INC-A,2024-12-27,133.3850,134.5000,133.5900,132.1600,2650366.0000,840,-1.4286,-0.1429,-1.5714,57739243957.1500
3,1256,ABNB:XNGS,AIRBNB INC-A,2024-12-26,135.3200,136.2700,134.2400,134.2400,2067513.0000,840,-1.7143,-1.1429,-2.8571,58576860158.8000
4,1256,ABNB:XNGS,AIRBNB INC-A,2024-12-24,134.9900,135.0700,134.3300,132.8300,1677060.0000,840,-2.0000,-2.2857,-4.2857,58434010884.1000


In [247]:
earnings = load_earnings(DATA_DIR)
earnings.head()

          | 0/3928

,Id,TickerId,EventDate,MarketTimeId
194,736622,1360,2020-01-29,2.0000
195,736623,1360,2020-04-29,2.0000
196,736624,1360,2020-07-22,2.0000
197,736625,1360,2020-10-21,2.0000
198,736626,1360,2021-02-03,2.0000


In [248]:
sentiment = load_sentiment(DATA_DIR)
sentiment.head()

          | 0/15471

,symbol,date,analystRatingsStrongBuy,analystRatingsBuy,analystRatingsHold,analystRatingsSell,analystRatingsStrongSell,ShortName_before,tickerId
3827,DAL,2019-01-01,7,11,3,0,0,DAL,15635
12862,TDY,2019-01-01,1,3,3,0,0,TDY,4219
5239,EXR,2019-01-01,3,5,10,1,0,EXR,2308
6528,GPN,2019-01-01,10,16,9,1,0,GPN,2564
7648,INVH,2019-01-01,6,9,2,0,0,INVH,11758


In [249]:
df_all = merge_panel(stock_data, earnings, sentiment)
df_all

          | 0/242507

,TickerId,ShortName,InstrumentName,Date,LastPrice,High,Open,Low,Volume,CurrencyId,QScoreTrendLong,QScoreSpeedLong,QScoreLong,MarketCap,Id,EventDate,MarketTimeId,Symbol,Exchange,symbol,date,analystRatingsStrongBuy,analystRatingsBuy,analystRatingsHold,analystRatingsSell,analystRatingsStrongSell,ShortName_before,tickerId
0,1256,ABNB:XNGS,AIRBNB INC-A,2024-12-31,131.4100,133.1900,132.4600,131.2700,2312564.0000,840,-0.8571,0.5714,-0.2857,56884312691.9000,NaN,NaT,NaN,ABNB,XNGS,ABNB,2024-12-01,2.0000,9.0000,28.0000,5.0000,3.0000,ABNB,1256.0000
1,1256,ABNB:XNGS,AIRBNB INC-A,2024-12-30,131.8100,132.6100,131.3100,129.8500,2458835.0000,840,-1.1429,0.4286,-0.7143,57057463327.9000,NaN,NaT,NaN,ABNB,XNGS,ABNB,2024-12-01,2.0000,9.0000,28.0000,5.0000,3.0000,ABNB,1256.0000
2,1256,ABNB:XNGS,AIRBNB INC-A,2024-12-27,133.3850,134.5000,133.5900,132.1600,2650366.0000,840,-1.4286,-0.1429,-1.5714,57739243957.1500,NaN,NaT,NaN,ABNB,XNGS,ABNB,2024-12-01,2.0000,9.0000,28.0000,5.0000,3.0000,ABNB,1256.0000
3,1256,ABNB:XNGS,AIRBNB INC-A,2024-12-26,135.3200,136.2700,134.2400,134.2400,2067513.0000,840,-1.7143,-1.1429,-2.8571,58576860158.8000,NaN,NaT,NaN,ABNB,XNGS,ABNB,2024-12-01,2.0000,9.0000,28.0000,5.0000,3.0000,ABNB,1256.0000
4,1256,ABNB:XNGS,AIRBNB INC-A,2024-12-24,134.9900,135.0700,134.3300,132.8300,1677060.0000,840,-2.0000,-2.2857,-4.2857,58434010884.1000,NaN,NaT,NaN,ABNB,XNGS,ABNB,2024-12-01,2.0000,9.0000,28.0000,5.0000,3.0000,ABNB,1256.0000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242502,15733,ANET:XNYS,"Arista Networks, Inc.",2020-01-08,51.7450,51.9475,51.2125,51.1575,2695700.0000,840,3.2857,1.2857,4.5714,64987654447.0950,NaN,NaT,NaN,ANET,XNYS,ANET,2020-01-01,9.0000,15.0000,13.0000,1.0000,0.0000,ANET,15733.0000
242503,15733,ANET:XNYS,"Arista Networks, Inc.",2020-01-07,51.2125,51.3575,50.7150,50.3250,2133496.0000,840,3.5714,0.7143,4.2857,64318876285.0875,NaN,NaT,NaN,ANET,XNYS,ANET,2020-01-01,9.0000,15.0000,13.0000,1.0000,0.0000,ANET,15733.0000
242504,15733,ANET:XNYS,"Arista Networks, Inc.",2020-01-06,50.7150,50.7925,49.6975,49.5000,2065432.0000,840,3.8571,0.7143,4.5714,63694055373.1650,NaN,NaT,NaN,ANET,XNYS,ANET,2020-01-01,9.0000,15.0000,13.0000,1.0000,0.0000,ANET,15733.0000
242505,15733,ANET:XNYS,"Arista Networks, Inc.",2020-01-03,50.2125,50.9575,50.0000,49.6200,3067648.0000,840,4.4286,1.2857,5.7143,63062954854.0875,NaN,NaT,NaN,ANET,XNYS,ANET,2020-01-01,9.0000,15.0000,13.0000,1.0000,0.0000,ANET,15733.0000


In [250]:
summary = df_all.describe(include="all").T
summary["dtype"] = df_all.dtypes
summary

,count,unique,top,freq,mean,min,25%,50%,75%,max,std,dtype
TickerId,242507.0000,NaN,NaN,NaN,5742.7565,1256.0000,2420.0000,3558.0000,11697.0000,15733.0000,4489.9229,int64
ShortName,242507,199,ANET:XNYS,2516,NaN,NaN,NaN,NaN,NaN,NaN,NaN,string
InstrumentName,242507,200,ARTHUR J GALLAGH,1258,NaN,NaN,NaN,NaN,NaN,NaN,NaN,string
Date,242507,NaN,NaN,NaN,2022-07-14 10:35:13.591772672,2020-01-02 00:00:00,2021-04-20 00:00:00,2022-07-19 00:00:00,2023-10-13 00:00:00,2024-12-31 00:00:00,NaN,datetime64[ns]
LastPrice,242507.0000,NaN,NaN,NaN,182.1043,1.5980,50.2231,98.8646,197.8903,9924.4000,433.6117,float64
High,242507.0000,NaN,NaN,NaN,184.5572,1.6990,50.8788,100.2585,200.6200,9964.7700,438.8381,float64
Open,242507.0000,NaN,NaN,NaN,182.0937,1.5820,50.2125,98.8679,198.0000,9914.1700,433.5253,float64
Low,242507.0000,NaN,NaN,NaN,179.5967,1.5760,49.5103,97.4500,195.2878,9794.0000,428.4314,float64
Volume,242507.0000,NaN,NaN,NaN,4748486.1069,100.0000,639752.0000,1454965.5000,3651269.5000,914081370.0000,14414255.2337,float64
CurrencyId,242507.0000,NaN,NaN,NaN,836.2946,124.0000,840.0000,840.0000,840.0000,840.0000,51.3744,int64


## 2. Aligned Panel And Benchmark

Build `df_align` with `src.features.build_aligned_panel` (uses `src.sessions` + `src.benchmark`).


In [251]:
from src.features import build_aligned_panel, build_event_table
from src.report import print_event_report


In [252]:
df_align = build_aligned_panel(df_all, data_dir=DATA_DIR)
df_align.head()

          | 0/242507

          | 0/1258

,TickerId,ShortName,InstrumentName,Date,LastPrice,High,Open,Low,Volume,CurrencyId,QScoreTrendLong,QScoreSpeedLong,QScoreLong,MarketCap,Id,EventDate,MarketTimeId,Symbol,Exchange,analystRatingsStrongBuy,analystRatingsBuy,analystRatingsHold,analystRatingsSell,analystRatingsStrongSell,BenchmarkAdjClose
0,1256,ABNB:XNGS,AIRBNB INC-A,2020-12-10,144.7100,165.0000,146.0000,141.2500,70447537.0000,840,NaN,NaN,NaN,59176032671.1700,NaN,NaT,NaN,ABNB,XNGS,NaN,NaN,NaN,NaN,NaN,338.2389
1,1256,ABNB:XNGS,AIRBNB INC-A,2020-12-11,139.2500,151.5000,146.5500,135.1000,26980830.0000,840,NaN,NaN,NaN,56943283459.7500,NaN,NaT,NaN,ABNB,XNGS,NaN,NaN,NaN,NaN,NaN,337.8423
2,1256,ABNB:XNGS,AIRBNB INC-A,2020-12-14,130.0000,135.3000,135.0000,125.1601,16966089.0000,840,NaN,NaN,NaN,53160695510.0000,NaN,NaT,NaN,ABNB,XNGS,NaN,NaN,NaN,NaN,NaN,336.3296
3,1256,ABNB:XNGS,AIRBNB INC-A,2020-12-15,124.8000,127.6000,126.6900,121.5000,10914432.0000,840,NaN,NaN,NaN,51034267689.6000,NaN,NaT,NaN,ABNB,XNGS,NaN,NaN,NaN,NaN,NaN,340.8766
4,1256,ABNB:XNGS,AIRBNB INC-A,2020-12-16,137.9900,142.0000,125.8300,124.9100,20409551.0000,840,NaN,NaN,NaN,56428033641.7300,NaN,NaT,NaN,ABNB,XNGS,NaN,NaN,NaN,NaN,NaN,341.4117


## 3. New Customized Data / Features

Attach market-level daily macro (`vix` / `tnx` / `dxy`) onto `df_align` by `Date`.

In [253]:
""" macro = pd.read_parquet(DATA_DIR / "macro_daily.parquet")
macro.head() """

' macro = pd.read_parquet(DATA_DIR / "macro_daily.parquet")\nmacro.head() '

In [254]:
""" df_align = df_align.merge(macro, on="Date", how="left")
df_align.head() """

' df_align = df_align.merge(macro, on="Date", how="left")\ndf_align.head() '

Build decision-day features on `df_align`: `mom_21d`, `vol_21d`, `log_mcap` (no lookahead; known at that close).

In [255]:
""" WINDOW = 21

df_align = df_align.sort_values(["TickerId", "Date"]).reset_index(drop=True)
price = df_align.groupby("TickerId", sort=False)["LastPrice"]
ret_1d = price.pct_change()
mcap = pd.to_numeric(df_align["MarketCap"], errors="coerce")

# Calculate the percentage change over the last 21 days 
df_align["mom_21d"] = price.pct_change(WINDOW)

# Calculate the standard deviation of the daily percentage changes over the last 21 days
df_align["vol_21d"] = (
    ret_1d.groupby(df_align["TickerId"], sort=False)
    .rolling(WINDOW, min_periods=WINDOW)
    .std(ddof=1)
    .reset_index(level=0, drop=True)
)

# Calculate the log of market cap
df_align["log_mcap"] = np.where(mcap > 0, np.log(mcap), np.nan)

df_align.head() """

' WINDOW = 21\n\ndf_align = df_align.sort_values(["TickerId", "Date"]).reset_index(drop=True)\nprice = df_align.groupby("TickerId", sort=False)["LastPrice"]\nret_1d = price.pct_change()\nmcap = pd.to_numeric(df_align["MarketCap"], errors="coerce")\n\n# Calculate the percentage change over the last 21 days \ndf_align["mom_21d"] = price.pct_change(WINDOW)\n\n# Calculate the standard deviation of the daily percentage changes over the last 21 days\ndf_align["vol_21d"] = (\n    ret_1d.groupby(df_align["TickerId"], sort=False)\n    .rolling(WINDOW, min_periods=WINDOW)\n    .std(ddof=1)\n    .reset_index(level=0, drop=True)\n)\n\n# Calculate the log of market cap\ndf_align["log_mcap"] = np.where(mcap > 0, np.log(mcap), np.nan)\n\ndf_align.head() '

## 4. Event Table

Build one-row-per-announcement `df` with `src.features.build_event_table` (reuses `df_align`); print descriptive stats with `src.report.print_event_report`.


In [256]:
df = build_event_table(
    df_all,
    data_dir=DATA_DIR,
    df_align=df_align,
    feature_columns=FEATURE_COLUMNS,
)
print_event_report(df, primary_target=TARGET, feature_columns=FEATURE_COLUMNS)


          | 0/3841


------------------------------------------------------------
df coverage
------------------------------------------------------------
rows=3837  cols=26  tickers=200
EventDate        2020-01-14 .. 2024-12-19
DecisionDate     2020-01-10 .. 2024-12-17
MarketTimeId
1    1700
2    2137

------------------------------------------------------------
df head
------------------------------------------------------------
   TickerId Symbol  ShortName   EventDate ReleaseDate  AnchorDate DecisionDate  MarketTimeId  ret_ann  ret_ann_mkt  ret_next  ret_next_mkt  ret_post  ret_post_mkt  dir_ann  vol_ann  vol_post  QScoreTrendLong  QScoreSpeedLong  QScoreLong  analystRatingsStrongBuy  analystRatingsBuy  analystRatingsHold  analystRatingsSell  analystRatingsStrongSell      Id
0      1256   ABNB  ABNB:XNGS  2021-02-25  2021-02-25  2021-02-26   2021-02-24             2  -0.0189      -0.0133    0.1334        0.1386   -0.1286       -0.1372       -1   0.1190    0.0240           2.4286          -0.5714      

## 5. Preprocess (Train / Val / Test)

Chronological split and train-only impute/scale via `src.preprocess.run_preprocess`; report with `src.report.print_preprocess_report`.


In [257]:
from src.preprocess import run_preprocess
from src.report import print_preprocess_report

In [258]:
result = run_preprocess(
    df,
    target=TARGET,
    feature_columns=FEATURE_COLUMNS,
    split_column=SPLIT_COLUMN,
    train_end=TRAIN_END,
    val_end=VAL_END,
    impute_strategy=IMPUTE_STRATEGY,
    standardize=STANDARDIZE,
)
print_preprocess_report(result, target=TARGET, split_column=SPLIT_COLUMN)
train_y_mean = float(result.folds["train"].y.to_numpy(dtype=float).mean())



------------------------------------------------------------
folds
------------------------------------------------------------
train n= 2266  2020-01-10 .. 2022-12-29
val   n=  775  2023-01-03 .. 2023-12-15
test  n=  793  2024-01-02 .. 2024-12-17
train_y_mean 0.007295513687746245

------------------------------------------------------------
target summary
------------------------------------------------------------
target=ret_ann_mkt  events_in=3837  dropped_target=3
       count  unique     min    max   mean    std  missing_n missing_rate
fold                                                                     
train   2266    2254 -0.4649 0.4845 0.0073 0.0732          0        0.00%
val      775     771 -0.3589 0.5867 0.0095 0.0828          0        0.00%
test     793     789 -0.4037 0.7185 0.0107 0.0917          0        0.00%

------------------------------------------------------------
train X head
------------------------------------------------------------
   QScoreTrendLong  

## 5+. Custom Model (demo-only)

Enable with `"custom"` in `FAMILIES` (§0a). Collaborator blocks map to main-family roles:

| Block | Role (like) | Model id |
| ----- | ----------- | -------- |
| **5+a** | `lin` | `elasticnet` |
| **5+b** | `tr` | `extratrees` |
| **5+c** | `nn` | `mlp_linear` |

Edit only 5+a/b/c. **Wire-up** is infrastructure.


In [259]:
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.linear_model import ElasticNet
from sklearn.neural_network import MLPRegressor

from src.preprocess import PreprocessResult
from src.train import FitBundle

# Which custom models to run (comment a line to skip that lin/tr/nn slot).
CUSTOM_MODEL_TYPES = (
    # "elasticnet",   
    # "extratrees",   
    # "mlp_linear",   
)
CUSTOM_STANDARDIZE = True

# Filled by 5+a / 5+b / 5+c; do not edit by hand.
_CUSTOM_FITTERS = {}
_CUSTOM_PARAMS = {}


### 5+a. `lin` slot → `elasticnet`

Custom stand-in for the **linear** family. Edit only this block.


In [260]:
""" # --- collaborator edit: lin / elasticnet ---
ELASTICNET = {
    "ALPHA": 1.0,
    "L1_RATIO": 0.5,
}


def fit_elasticnet(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    alpha = float(ELASTICNET["ALPHA"])
    l1_ratio = float(ELASTICNET["L1_RATIO"])
    model = ElasticNet(
        alpha=alpha,
        l1_ratio=l1_ratio,
        max_iter=10_000,
        random_state=RANDOM_STATE,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={"alpha": alpha, "l1_ratio": l1_ratio},
    )


_CUSTOM_FITTERS["elasticnet"] = fit_elasticnet
_CUSTOM_PARAMS.update(ELASTICNET) """


' # --- collaborator edit: lin / elasticnet ---\nELASTICNET = {\n    "ALPHA": 1.0,\n    "L1_RATIO": 0.5,\n}\n\n\ndef fit_elasticnet(result: PreprocessResult) -> FitBundle:\n    train = result.folds["train"]\n    alpha = float(ELASTICNET["ALPHA"])\n    l1_ratio = float(ELASTICNET["L1_RATIO"])\n    model = ElasticNet(\n        alpha=alpha,\n        l1_ratio=l1_ratio,\n        max_iter=10_000,\n        random_state=RANDOM_STATE,\n    )\n    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))\n    return FitBundle(\n        model=model,\n        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),\n        summary_extras={"alpha": alpha, "l1_ratio": l1_ratio},\n    )\n\n\n_CUSTOM_FITTERS["elasticnet"] = fit_elasticnet\n_CUSTOM_PARAMS.update(ELASTICNET) '

### 5+b. `tr` slot → `extratrees`

Custom stand-in for the **tree** family. Edit only this block.


In [261]:
""" # --- collaborator edit: tr / extratrees ---
EXTRATREES = {
    "N_ESTIMATORS": 200,
    "MAX_DEPTH": 6,
    "MIN_SAMPLES_LEAF": 20,
}


def fit_extratrees(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    model = ExtraTreesRegressor(
        n_estimators=int(EXTRATREES["N_ESTIMATORS"]),
        max_depth=int(EXTRATREES["MAX_DEPTH"]),
        min_samples_leaf=int(EXTRATREES["MIN_SAMPLES_LEAF"]),
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={
            "n_estimators": int(EXTRATREES["N_ESTIMATORS"]),
            "max_depth": int(EXTRATREES["MAX_DEPTH"]),
        },
    )


_CUSTOM_FITTERS["extratrees"] = fit_extratrees
_CUSTOM_PARAMS.update(EXTRATREES) """


' # --- collaborator edit: tr / extratrees ---\nEXTRATREES = {\n    "N_ESTIMATORS": 200,\n    "MAX_DEPTH": 6,\n    "MIN_SAMPLES_LEAF": 20,\n}\n\n\ndef fit_extratrees(result: PreprocessResult) -> FitBundle:\n    train = result.folds["train"]\n    model = ExtraTreesRegressor(\n        n_estimators=int(EXTRATREES["N_ESTIMATORS"]),\n        max_depth=int(EXTRATREES["MAX_DEPTH"]),\n        min_samples_leaf=int(EXTRATREES["MIN_SAMPLES_LEAF"]),\n        random_state=RANDOM_STATE,\n        n_jobs=-1,\n    )\n    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))\n    return FitBundle(\n        model=model,\n        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),\n        summary_extras={\n            "n_estimators": int(EXTRATREES["N_ESTIMATORS"]),\n            "max_depth": int(EXTRATREES["MAX_DEPTH"]),\n        },\n    )\n\n\n_CUSTOM_FITTERS["extratrees"] = fit_extratrees\n_CUSTOM_PARAMS.update(EXTRATREES) '

### 5+c. `nn` slot → `mlp_linear`

Custom stand-in for the **neural** family. Edit only this block.
(`activation="identity"` ≈ linear map through a hidden layer.)


In [262]:
""" # --- collaborator edit: nn / mlp_linear ---
MLP_LINEAR = {
    "MLP_HIDDEN": (32,),
    "MLP_MAX_ITER": 300,
}


def fit_mlp_linear(result: PreprocessResult) -> FitBundle:
    train = result.folds["train"]
    hidden = tuple(MLP_LINEAR["MLP_HIDDEN"])
    model = MLPRegressor(
        hidden_layer_sizes=hidden,
        activation="identity",
        solver="adam",
        max_iter=int(MLP_LINEAR["MLP_MAX_ITER"]),
        random_state=RANDOM_STATE,
    )
    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))
    return FitBundle(
        model=model,
        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),
        summary_extras={"hidden": str(hidden)},
    )


_CUSTOM_FITTERS["mlp_linear"] = fit_mlp_linear
_CUSTOM_PARAMS.update(MLP_LINEAR) """


' # --- collaborator edit: nn / mlp_linear ---\nMLP_LINEAR = {\n    "MLP_HIDDEN": (32,),\n    "MLP_MAX_ITER": 300,\n}\n\n\ndef fit_mlp_linear(result: PreprocessResult) -> FitBundle:\n    train = result.folds["train"]\n    hidden = tuple(MLP_LINEAR["MLP_HIDDEN"])\n    model = MLPRegressor(\n        hidden_layer_sizes=hidden,\n        activation="identity",\n        solver="adam",\n        max_iter=int(MLP_LINEAR["MLP_MAX_ITER"]),\n        random_state=RANDOM_STATE,\n    )\n    model.fit(train.X.to_numpy(dtype=float), np.asarray(train.y, dtype=float))\n    return FitBundle(\n        model=model,\n        predict=lambda X, m=model: m.predict(X.to_numpy(dtype=float)),\n        summary_extras={"hidden": str(hidden)},\n    )\n\n\n_CUSTOM_FITTERS["mlp_linear"] = fit_mlp_linear\n_CUSTOM_PARAMS.update(MLP_LINEAR) '

#### Wire-up

Not a lin/tr/nn model block. Errors if `CUSTOM_MODEL_TYPES` names a model that
was not registered by 5+a/b/c; then builds `fit_custom` / `CUSTOM`.


In [263]:
_unknown = [name for name in CUSTOM_MODEL_TYPES if name not in _CUSTOM_FITTERS]
if _unknown:
    raise ValueError(
        "CUSTOM_MODEL_TYPES names are not registered in §5+a/b/c "
        f"(no fit_* block): {_unknown}. "
        f"Registered: {sorted(_CUSTOM_FITTERS)}."
    )
_unused = sorted(set(_CUSTOM_FITTERS) - set(CUSTOM_MODEL_TYPES))
if _unused:
    print("note: registered but not in CUSTOM_MODEL_TYPES:", _unused)


def fit_custom(model_type: str, result: PreprocessResult) -> FitBundle:
    try:
        fitter = _CUSTOM_FITTERS[model_type]
    except KeyError as exc:
        raise ValueError(
            f"Unknown custom model_type={model_type!r}; "
            f"registered={sorted(_CUSTOM_FITTERS)}."
        ) from exc
    return fitter(result)


CUSTOM = {
    "MODEL_TYPES": tuple(CUSTOM_MODEL_TYPES),
    "STANDARDIZE": bool(CUSTOM_STANDARDIZE),
    **_CUSTOM_PARAMS,
}

ACTIVE_FAMILIES = [
    ("custom", dict(CUSTOM)) if name == "custom" else (name, block)
    for name, block in ACTIVE_FAMILIES
]
if ACTIVE_FAMILIES and ACTIVE_FAMILIES[0][0] == "custom":
    STANDARDIZE = bool(CUSTOM["STANDARDIZE"])

print("fit_custom ready:", CUSTOM["MODEL_TYPES"])
print("registered:", sorted(_CUSTOM_FITTERS))


fit_custom ready: ()
registered: []


## 6. Fit And Predict

For each active family: `lin`/`tr`/`nn` via `apply_demo_block`; `custom` via
`build_demo_custom_spec` + §5+ `fit_custom`. Then re-preprocess and `run_one_model`.


In [264]:
from src.metrics import infer_task
from src.pipeline import run_one_model
from src.preprocess import run_preprocess
from src.train import apply_demo_block, build_demo_custom_spec

In [265]:
task = infer_task(TARGET)
if task != "regression":
    raise NotImplementedError(f"Demo supports regression only; got TARGET={TARGET!r}")

summary_rows = []
for family, block in ACTIVE_FAMILIES:
    if family == "custom":
        spec = build_demo_custom_spec(
            block,
            fit_one=fit_custom,
            feature_importance=FEATURE_IMPORTANCE,
            feature_importance_repeats=FEATURE_IMPORTANCE_REPEATS,
            random_state=RANDOM_STATE,
        )
    else:
        spec = apply_demo_block(
            family,
            block,
            feature_importance=FEATURE_IMPORTANCE,
            feature_importance_repeats=FEATURE_IMPORTANCE_REPEATS,
            random_state=RANDOM_STATE,
        )
    result_f = run_preprocess(
        df,
        target=TARGET,
        feature_columns=FEATURE_COLUMNS,
        split_column=SPLIT_COLUMN,
        train_end=TRAIN_END,
        val_end=VAL_END,
        impute_strategy=IMPUTE_STRATEGY,
        standardize=bool(block["STANDARDIZE"]),
    )
    train_y_mean_f = float(result_f.folds["train"].y.to_numpy(dtype=float).mean())
    print(f"\n### family={family}  standardize={block['STANDARDIZE']}  "
          f"models={list(spec.model_types)}")
    for model_type in spec.model_types:
        row, _predictions = run_one_model(
            spec,
            model_type=model_type,
            result=result_f,
            train_y_mean=train_y_mean_f,
            task=task,
        )
        summary_rows.append(row)

summary = pd.DataFrame(summary_rows)



### family=lin  standardize=True  models=['ols', 'ridge', 'lasso']

OLS

------------------------------------------------------------
METRICS
------------------------------------------------------------
          n           r2 (oos)        rmse (mean/zero)         mae (mean/zero)       ic rank_ic hit_rate
fold                                                                                                    
train  2266    0.0032 (0.0032)  0.0731 (0.0732/0.0735)  0.0525 (0.0526/0.0526)   0.0569  0.0253   0.5340
val     775  -0.0041 (-0.0034)  0.0829 (0.0828/0.0833)  0.0590 (0.0592/0.0593)  -0.0086  0.0225   0.5381
test    793  -0.0097 (-0.0082)  0.0922 (0.0918/0.0924)  0.0651 (0.0650/0.0653)  -0.0115  0.0059   0.5271

------------------------------------------------------------
QUINTILES
------------------------------------------------------------
fold   n     q1     q2      q3     q4     q5  top_minus_bottom
 val 775 0.0106 0.0064  0.0177 0.0034 0.0096           -0.0010
test 793 0.0

## 7. Summary

Compact cross-model comparison on the test fold (`print_summary_table`).


In [267]:
print(summary)

   model    alpha  n_test      r2  oos_r2   rmse    mae  rmse_train_mean  mae_train_mean  rmse_zero  mae_zero      ic  rank_ic  hit_rate  top_minus_bottom
0    ols   1.0000     793 -0.0097 -0.0082 0.0922 0.0651           0.0918          0.0650     0.0924    0.0653 -0.0115   0.0059    0.5271            0.0035
1  ridge 100.0000     793 -0.0089 -0.0075 0.0921 0.0651           0.0918          0.0650     0.0924    0.0653 -0.0127   0.0044    0.5284            0.0027
2  lasso   0.0100     793 -0.0014  0.0000 0.0918 0.0650           0.0918          0.0650     0.0924    0.0653     NaN      NaN    0.5309               NaN
